# MergeDeclaredLinks — nightly merges of works linked by a declaration (oxjob #1057, #1099)

Runs between `Locations_with_Types` and `Map_Work_Ids`. When a record declares a link to another record (a preprint to its
article, a file to its dataset) after both already sit on their own pinned works, MapWorkIds never re-resolves them: this step
merges those works every night. One step for every kind of link (Casey 2026-10-25, oxjob #1057 `MERGE_DECLARED_LINKS.md`):
each **class** has its own pair rule, winner and evidence holds; the holds' shared check, the tripwire, the execute and the
citation moves run once for all classes. It replaces `MergeDeclaredPreprints` (oxjob #1099), whose preprint rule is the
`preprint` class below, cell for cell.

| Class | Pairs | Winner | `merged_work_ids` source | Tripwire |
|---|---|---|---|---|
| `preprint` (oxjob #1099, 2026-10-08) | declared preprint ↔ article DOI pairs, typed (`preprint_link_pairs`) | the group's most-cited work, ties to the older id | `preprint_link_nightly` | 100,000 |
| `part_of` (oxjob #1057, 2026-10-23) | a DataCite `dataset` record IsPartOf a `dataset` record on another work | the top dataset of the chain | `part_of_nightly` | 50,000 |

**preprint** (unchanged from MergeDeclaredPreprints):

MapWorkIds' `crossref_preprint` tier joins a record to its declared
partner only when the link is there on the record's first night; most links are added later (the preprint server records the
article after it is out, and Crossref mirrors the claim onto the article as `has-preprint`), when both records already sit on
their own works and are pinned. This step merges those works every night (Casey 2026-10-08: "build it into the nightly").

- **Pairs** (view `openalex.works.preprint_link_pairs`, recreated here first; MapWorkIds' `crossref_preprint` tier reads its
  Crossref arm): declared by either Crossref record (`is-preprint-of` / `has-preprint` in `ids[]`), or by arXiv's DataCite record
  (`IsVersionOf` a journal DOI; arXiv adds it after publication, oxjob #1256's declared_version pairs); preprint = Crossref
  posted-content/preprint or a DataCite `preprint` (arXiv 10.48550), article = Crossref journal-article typed article / review /
  letter; both pinned, on two works that are in `openalex_works`.
- **Group** = an article's work + the works of every preprint it pairs with. **Winner = the most-cited member, ties to the older
  (lower) id.** A just-arrived article has no citations yet, so it joins the preprint's work (Casey 2026-10-08: the published work
  that comes in and points to a preprint just joins the preprint work); an established, cited article keeps its id (first-night
  backlog: the T2T genome, gnomAD, ARRIVE 2.0 articles would otherwise 404 onto their bioRxiv ids). The merged work presents
  as the article either way (CreateWorksBase prefers the published Crossref VoR, #589).
- **Holds** (evidence of a bad link): the article's work titled as a correction, typed other than article / review / letter, a
  year gap (article > 3 years after or > 1 before the preprint), or content that differs (titles differ and abstract token Jaccard
  < 0.3 or no abstract to compare: blind eval 2026-10-09, 0/100 wrong where content agrees; #1652 2026-10-14, 1/100 wrong
  + 0/15 top-cited in the 0.3-0.6 band, wrong links there are companion papers); arXiv-only:
  the preprint's work already shows another DOI. Mechanical: a preprint work paired with two article works, a work on both sides, a
  loser key held by a work outside the group. Any held pair holds its whole group.

**part_of**: a DataCite record typed `dataset` declares `IsPartOf` a DOI whose Crossref or DataCite record is typed `dataset`
(Casey 2026-10-23: datasets only; sub-datasets merge too, cited or not), minus its own DOI and any DOI it also names as a version
(left to #1581, as CreateWorksBase's part rule). Both pinned, on two live works. Winner = the top of the chain (a file inside a
sub-dataset lands on the outer dataset's work); a part whose parents reach one top merges there, one whose parents reach
different tops is held (`multi_parent`), a chain of 6+ steps or a loop is held (`deep_or_loop`); mechanical holds per loser:
`chained`, `loser_key_shared`. Same rule as the 2026-10-23 backlog wave (2,211,708 parts onto 284,241 datasets; blind 50/50),
plus one the backlog didn't need, since every part was then alone on its own work: a work that already holds a dataset and
its files moves only on the dataset's own declaration (a sub-dataset), never because one of its files names a second parent
(first test 2026-10-26: a Chandra observation in two data collections would have merged the collections).
Most new files arrive the same night as their dataset, so neither has a work when MapWorkIds sees them: both mint, and this step
merges them the next night. CreateWorksBase's part rule (walden 6e1379f7) gives the merged work the dataset's title and DOI.

- **Shared check:** a work that is a loser or a winner in two classes the same night is held in both (`cross_class`), with its
  whole group, so it is never merged two ways at once.
- **Execute** (as `MergeIdenticalKeyWorks` execute, once for all classes): audit → aliases of the losers' record keys onto the
  winner → drop the losers' map rows → unpin the losers → `merged_work_ids` (the class's source) → ledger. MapWorkIds then
  re-resolves the losers' records onto the winner in this run. Map rows go before pins, so a partial run leaves a loser's records
  pinned where they were, never re-resolving onto it.
- **Citations:** `work_references` rows citing a ledgered loser, and the loser's own reference rows, move to the winner (every
  loser in `declared_link_merges` and the old `preprint_link_merges` history).
- **Tripwire:** more executable losers than the class's cap fails the task before any write (a bug, not a backlog).
  `Merge_Declared_Links_Gate` passes either way, so MapWorkIds always runs.
- **Undo** (before the next nightly), per class: re-insert the run's `declared_link_merges_audit` pin / map rows of that `link`,
  delete its alias rows from `work_id_map`, delete the run's `merged_work_ids` rows (that class's source). The preprint step's
  own tables (`preprint_link_merges`, `_audit`, `_refs_audit`) stay as history; nothing writes them after this replaced it.

In [ ]:
CREATE OR REPLACE VIEW openalex.works.preprint_link_pairs
COMMENT 'oxjob #1099: declared preprint <-> published-article DOI pairs, typed. One definition for MapWorkIds (crossref_preprint tier, arm = crossref) and MergeDeclaredPreprints (both arms). Recreated nightly by MergeDeclaredPreprints.'
AS
-- Crossref: declared by either record (`is-preprint-of` on the preprint, `has-preprint` on the article; Crossref mirrors one
-- side's claim onto the other)
WITH cr_links AS (
  SELECT DISTINCT
         CASE WHEN x.relationship = 'is-preprint-of' THEN regexp_replace(l.native_id, '[^a-zA-Z0-9\./-]', '') ELSE regexp_replace(x.id, '[^a-zA-Z0-9\./-]', '') END AS pre,
         CASE WHEN x.relationship = 'is-preprint-of' THEN regexp_replace(x.id, '[^a-zA-Z0-9\./-]', '') ELSE regexp_replace(l.native_id, '[^a-zA-Z0-9\./-]', '') END AS pub
  FROM openalex.works.locations_w_types l LATERAL VIEW explode(l.ids) e AS x
  WHERE l.provenance = 'crossref' AND EXISTS(l.ids, y -> y.relationship IN ('is-preprint-of', 'has-preprint'))
    AND x.relationship IN ('is-preprint-of', 'has-preprint')
),
-- arXiv's DataCite record declaring IsVersionOf its journal DOI (oxjob #1256's declared_version pairs); arXiv adds the DOI
-- after the article is out, so most of these appear when both records are already pinned
dc_links AS (
  SELECT DISTINCT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS pre,
         regexp_replace(regexp_replace(LOWER(x.id), '^https?://(dx\\.)?doi\\.org/', ''), '[^a-zA-Z0-9\./-]', '') AS pub
  FROM openalex.works.locations_w_types l LATERAL VIEW explode(l.ids) e AS x
  WHERE l.provenance = 'datacite' AND l.type = 'preprint' AND l.native_id LIKE '10.48550/%'
    AND x.relationship = 'IsVersionOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND LOWER(x.id) NOT LIKE '%10.48550/%'
),
links AS (
  SELECT pre, pub, CASE WHEN BOOL_OR(arm = 'crossref') THEN 'crossref' ELSE 'datacite' END AS arm
  FROM (SELECT pre, pub, 'crossref' AS arm FROM cr_links UNION ALL SELECT pre, pub, 'datacite' FROM dc_links)
  WHERE pre <> pub
  GROUP BY pre, pub
)
-- typed: preprint = Crossref posted-content/preprint or a DataCite preprint (AACR registers ~204K Figshare supplements as
-- `is-preprint-of` with subtype `other`); article = Crossref journal-article typed article / review / letter
SELECT k.pre, k.pub, k.arm FROM links k
LEFT SEMI JOIN openalex.works.locations_w_types a
  ON a.provenance IN ('crossref', 'datacite') AND regexp_replace(LOWER(a.native_id), '[^a-zA-Z0-9\./-]', '') = k.pre
 AND ((a.provenance = 'crossref' AND a.raw_type = 'posted-content' AND a.type = 'preprint')
   OR (a.provenance = 'datacite' AND a.type = 'preprint'))
LEFT SEMI JOIN openalex.works.locations_w_types b
  ON b.provenance = 'crossref' AND regexp_replace(b.native_id, '[^a-zA-Z0-9\./-]', '') = k.pub
 AND b.raw_type = 'journal-article' AND b.type IN ('article', 'review', 'letter')

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.declared_link_merges (
  run_ts TIMESTAMP, link STRING, loser_work_id BIGINT, winner_work_id BIGINT, group_work_id BIGINT, n_pairs INT, signals STRING)
CLUSTER BY (loser_work_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.declared_link_merges_audit (
  run_ts TIMESTAMP, link STRING, kind STRING, loser_work_id BIGINT, winner_work_id BIGINT,
  provenance STRING, native_id_namespace STRING, native_id STRING, work_id_source STRING,
  openalex_created_dt DATE, openalex_updated_dt TIMESTAMP, seeded_dt DATE, seeded_from STRING,
  doi STRING, pmid STRING, arxiv STRING, title_author STRING, created_date DATE, updated_date TIMESTAMP)
CLUSTER BY (run_ts, loser_work_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.declared_link_merges_refs_audit (
  run_ts TIMESTAMP, side STRING, citing_work_id BIGINT, native_id STRING, native_id_namespace STRING, ref_ind BIGINT,
  old_id BIGINT, new_id BIGINT)

## Class `preprint` (oxjob #1099): pairs on two works, evidence holds, groups and winners
The MergeDeclaredPreprints cells, unchanged.

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_pairs AS
WITH links AS (SELECT pre, pub, arm = 'crossref' AS cr_declared FROM openalex.works.preprint_link_pairs),
ld AS (SELECT pre AS d FROM links UNION SELECT pub FROM links),
lt AS (
  SELECT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS d, l.provenance, r.work_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN ld ON ld.d = regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '')
  JOIN openalex.works.location_work_ids r
    ON r.provenance = l.provenance AND r.native_id_namespace = l.native_id_namespace AND r.native_id = l.native_id
  WHERE l.provenance IN ('crossref', 'datacite') AND r.work_id IS NOT NULL
),
pw AS (
  SELECT DISTINCT k.pre AS pre_doi, k.pub AS pub_doi, k.cr_declared, a.work_id AS pre_work, b.work_id AS pub_work
  FROM links k JOIN lt a ON a.d = k.pre JOIN lt b ON b.d = k.pub AND b.provenance = 'crossref'
  WHERE a.work_id <> b.work_id
),
wp AS (SELECT DISTINCT pre_work, pub_work FROM pw),
ids AS (SELECT pre_work AS id FROM wp UNION SELECT pub_work FROM wp),
wsel AS (
  SELECT w.id, w.publication_year AS yr, w.type, w.title, COALESCE(w.cited_by_count, 0) AS cites, w.abstract,
         regexp_replace(lower(w.title), '[^\\p{L}\\p{N}]', '') AS tn,
         regexp_replace(lower(w.doi), '^https?://(dx\\.)?doi\\.org/', '') AS doi
  FROM openalex.works.openalex_works w LEFT SEMI JOIN ids ON ids.id = w.id
),
-- abstract tokens as rows, Jaccard by join: array set functions over per-work token lists gave run-varying answers in both
-- directions on the warehouse (2026-10-07 arrays_overlap; 2026-10-09 size(array_intersect): 61-193 pairs flipped per run)
atok AS (
  SELECT DISTINCT id, t FROM wsel
  LATERAL VIEW explode(split(regexp_replace(lower(COALESCE(abstract, '')), '[^a-z0-9 ]', ' '), ' +')) e AS t
  WHERE length(t) > 3
),
nab AS (SELECT id, COUNT(*) AS n FROM atok GROUP BY id),
ain AS (SELECT p.pre_work, p.pub_work, COUNT(*) AS inter FROM wp p JOIN atok a ON a.id = p.pre_work JOIN atok b ON b.id = p.pub_work AND b.t = a.t GROUP BY 1, 2),
wf AS (
  SELECT s.id, s.yr, s.type, s.title, s.cites, s.tn, s.doi, COALESCE(na.n, 0) AS n_abs
  FROM wsel s LEFT JOIN nab na ON na.id = s.id
),
r AS (
  SELECT p.*, x.yr AS pre_yr, y.yr AS pub_yr, y.type AS pub_type, x.cites AS pre_cites, y.cites AS pub_cites,
         LEFT(x.title, 200) AS pre_title, LEFT(y.title, 200) AS pub_title, y.title AS pub_title_full,
         -- oxjob #1676: a missing or empty title never agrees and never skips the content_differs hold. Plain x.tn = y.tn was
         -- NULL for an untitled side (NOT NULL is NULL, so the hold was skipped and the pair merged unchecked) and TRUE for two ''
         COALESCE(x.tn = y.tn AND x.tn <> '', FALSE) AS title_same,
         x.doi AS pre_work_doi,
         CASE WHEN x.n_abs >= 20 AND y.n_abs >= 20 THEN COALESCE(ai.inter, 0) / (x.n_abs + y.n_abs - COALESCE(ai.inter, 0)) END AS abs_jaccard
  FROM pw p JOIN wf x ON x.id = p.pre_work JOIN wf y ON y.id = p.pub_work
  LEFT JOIN ain ai ON ai.pre_work = p.pre_work AND ai.pub_work = p.pub_work
)
SELECT pre_doi, pub_doi, pre_work, pub_work, pre_yr, pub_yr, pub_type, pre_cites, pub_cites, pre_title, pub_title,
       CASE WHEN cr_declared THEN 'crossref' ELSE 'datacite' END AS arm, abs_jaccard,
       concat_ws('+', 'declared', CASE WHEN title_same THEN 'title' END, CASE WHEN abs_jaccard >= 0.3 THEN 'abstract' END) AS signals,
       CASE WHEN pub_title_full RLIKE '(?i)^\\s*(metadata correction|correction|erratum|corrigendum|addendum)([^a-z]|$)' THEN 'correction_notice'
            WHEN pub_type NOT IN ('article', 'review', 'letter') THEN 'article_type'
            WHEN pre_yr IS NOT NULL AND pub_yr IS NOT NULL AND (pub_yr - pre_yr > 3 OR pub_yr - pre_yr < -1) THEN 'year_gap'
            -- the declaration merges when the content agrees: same normalized title, or abstracts agree (token Jaccard >= 0.3). Blind
            -- eval 2026-10-09 (240 pairs): 0/100 wrong where content agrees; every wrong link (JMIR's 2018 off-by-N deposits, OSF
            -- REMOVED/WITHDRAWN placeholders, mistyped DOIs) had a different title and abstracts < 0.6 or none. The 0.3-0.6 band
            -- (#1652, 2026-10-14): 1/100 wrong + 0/15 top-cited; its wrong links are companion papers (0.33, 0.42). An author-name check
            -- let ~0.7 % wrong through (common surnames, same-group papers) and held collaboration papers ("ATLAS Collaboration").
            WHEN NOT title_same AND COALESCE(abs_jaccard, 0) < 0.3 THEN 'content_differs'
            -- arXiv-only: the preprint's work already presents another DOI (a journal record glued on): a three-way case
            WHEN NOT cr_declared AND pre_work_doi IS NOT NULL AND pre_work_doi <> pre_doi AND pre_work_doi NOT LIKE '10.48550/%' THEN 'pre_work_other_doi'
       END AS evidence_hold
FROM r

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_losers AS
WITH multi AS (SELECT pre_work FROM openalex.works.preprint_link_merge_pairs GROUP BY pre_work HAVING COUNT(DISTINCT pub_work) > 1),
both_sides AS (SELECT DISTINCT a.pub_work AS w FROM openalex.works.preprint_link_merge_pairs a
               JOIN openalex.works.preprint_link_merge_pairs b ON b.pre_work = a.pub_work),
pr AS (
  SELECT p.*,
         COALESCE(p.evidence_hold,
                  CASE WHEN m.pre_work IS NOT NULL THEN 'multi_article'
                       WHEN c1.w IS NOT NULL OR c2.w IS NOT NULL THEN 'chained' END) AS pair_hold
  FROM openalex.works.preprint_link_merge_pairs p
  LEFT JOIN multi m ON m.pre_work = p.pre_work
  LEFT JOIN both_sides c1 ON c1.w = p.pub_work
  LEFT JOIN both_sides c2 ON c2.w = p.pre_work
),
mc AS (SELECT pub_work AS g, pre_work AS w, pre_cites AS cites FROM pr UNION SELECT pub_work, pub_work, pub_cites FROM pr),
grp AS (
  SELECT p.pub_work AS group_work_id, w.winner_work_id, COUNT(*) AS n_pairs, MIN(p.pair_hold) AS group_hold, MIN(p.signals) AS signals
  FROM pr p
  JOIN (SELECT g, MAX_BY(w, named_struct('cites', cites, 'older', -w)) AS winner_work_id FROM mc GROUP BY g) w ON w.g = p.pub_work
  GROUP BY p.pub_work, w.winner_work_id
),
members AS (
  SELECT DISTINCT g.group_work_id, g.winner_work_id, m.w AS work_id
  FROM grp g JOIN mc m ON m.g = g.group_work_id
),
losers AS (
  SELECT m.group_work_id, m.winner_work_id, m.work_id AS loser_work_id, g.n_pairs, g.signals, g.group_hold
  FROM members m JOIN grp g ON g.group_work_id = m.group_work_id
  WHERE m.work_id <> m.winner_work_id
),
-- the losers' record keys, as MapWorkIds will look them up after the unpin
lrec AS (
  SELECT lo.group_work_id, lo.loser_work_id, l.merge_key AS mk
  FROM losers lo
  JOIN openalex.works.location_work_ids r ON r.work_id = lo.loser_work_id
  JOIN openalex.works.locations_w_types l
    ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
  WHERE lo.group_hold IS NULL
),
lkeys AS (
  SELECT DISTINCT x.group_work_id, x.loser_work_id, kv.col, kv.k
  FROM lrec x LATERAL VIEW explode(map('doi', NULLIF(x.mk.doi, ''), 'pmid', NULLIF(x.mk.pmid, ''),
                                       'arxiv', CASE WHEN x.mk.arxiv IN ('arXiv:', '') THEN NULL ELSE x.mk.arxiv END,
                                       'title_author', NULLIF(x.mk.title_author, ''))) kv AS col, k
  WHERE kv.k IS NOT NULL
),
keyless AS (
  SELECT DISTINCT group_work_id, loser_work_id FROM lrec
  WHERE NULLIF(mk.doi, '') IS NULL AND NULLIF(mk.pmid, '') IS NULL AND (mk.arxiv IS NULL OR mk.arxiv IN ('arXiv:', ''))
),
kin AS (SELECT DISTINCT col, k FROM lkeys),
holders AS (
  SELECT 'doi' AS col, m.doi AS k, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'doi' AND kin.k = m.doi
  UNION ALL SELECT 'pmid', m.pmid, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'pmid' AND kin.k = m.pmid
  UNION ALL SELECT 'arxiv', m.arxiv, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'arxiv' AND kin.k = m.arxiv
  UNION ALL SELECT 'title_author', m.title_author, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'title_author' AND kin.k = m.title_author
),
-- a key held by a work outside the group; a shared title key only counts for a loser with a doi / pmid / arxiv-less record
shared AS (
  SELECT DISTINCT lk.group_work_id FROM lkeys lk JOIN holders h ON h.col = lk.col AND h.k = lk.k
  WHERE NOT EXISTS (SELECT 1 FROM members m WHERE m.group_work_id = lk.group_work_id AND m.work_id = h.id)
    AND (lk.col <> 'title_author' OR EXISTS (SELECT 1 FROM keyless kl WHERE kl.group_work_id = lk.group_work_id AND kl.loser_work_id = lk.loser_work_id))
)
SELECT lo.group_work_id, lo.winner_work_id, lo.loser_work_id, lo.n_pairs, lo.signals,
       COALESCE(lo.group_hold, CASE WHEN sh.group_work_id IS NOT NULL THEN 'loser_key_shared' END) AS hold_reason,
       current_timestamp() AS run_ts
FROM losers lo LEFT JOIN shared sh ON sh.group_work_id = lo.group_work_id

In [ ]:
SELECT hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.preprint_link_merge_losers GROUP BY 1 ORDER BY 2 DESC

## Class `part_of` (oxjob #1057): parts on another work, one row per loser, mechanical holds

In [ ]:
CREATE OR REPLACE TABLE openalex.works.part_of_merge_edges AS
-- a DataCite dataset record's declared IsPartOf DOIs, minus its own and any it also names as a version (CreateWorksBase's
-- part_of_dois, oxjob #1057)
WITH decl AS (
  SELECT DISTINCT d.provenance, d.native_id_namespace, d.native_id, p AS parent_doi
  FROM (
    SELECT provenance, native_id_namespace, native_id,
           ARRAY_EXCEPT(
             ARRAY_DISTINCT(TRANSFORM(FILTER(ids, x -> x.relationship = 'IsPartOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND x.id IS NOT NULL),
               x -> regexp_replace(LOWER(TRIM(x.id)), '^(https?://(dx[.])?doi[.]org/|doi:)', ''))),
             ARRAY_UNION(ARRAY(LOWER(native_id)),
               TRANSFORM(FILTER(ids, x -> x.relationship IN ('IsVersionOf', 'HasVersion') AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND x.id IS NOT NULL),
                 x -> regexp_replace(LOWER(TRIM(x.id)), '^(https?://(dx[.])?doi[.]org/|doi:)', '')))) AS parents
    FROM openalex.works.locations_w_types
    WHERE provenance = 'datacite' AND type = 'dataset' AND EXISTS(ids, x -> x.relationship = 'IsPartOf')
  ) d LATERAL VIEW explode(d.parents) e AS p
),
pd AS (SELECT DISTINCT parent_doi FROM decl),
-- the parent DOI's records typed dataset (Crossref or DataCite), pinned
par AS (
  SELECT DISTINCT LOWER(l.native_id) AS doi, r.work_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN pd ON pd.parent_doi = LOWER(l.native_id)
  JOIN openalex.works.location_work_ids r
    ON r.provenance = l.provenance AND r.native_id_namespace = l.native_id_namespace AND r.native_id = l.native_id
  WHERE l.provenance IN ('crossref', 'datacite') AND l.native_id_namespace = 'doi' AND l.type = 'dataset' AND r.work_id IS NOT NULL
),
-- each declaring record's work and its parent's work
rp AS (
  SELECT DISTINCT LOWER(d.native_id) AS rec_doi, r.work_id AS c, p.work_id AS p, d.parent_doi
  FROM decl d
  JOIN openalex.works.location_work_ids r
    ON r.provenance = d.provenance AND r.native_id_namespace = d.native_id_namespace AND r.native_id = d.native_id
  JOIN par p ON p.doi = d.parent_doi
  WHERE r.work_id IS NOT NULL
)
-- only works with a part or a parent on another work (a few thousand), with every declaration on them
SELECT rec_doi, c, p, parent_doi
FROM (SELECT rp.*, MAX(CAST(c <> p AS INT)) OVER (PARTITION BY c) AS moves FROM rp)
WHERE moves = 1

In [ ]:
CREATE OR REPLACE TABLE openalex.works.part_of_merge_pairs AS
WITH rp AS (SELECT rec_doi, c, p, parent_doi FROM openalex.works.part_of_merge_edges),
-- parents already on the same work as their parts: that work is a dataset holding its files
seated AS (SELECT DISTINCT c AS w, parent_doi FROM rp WHERE c = p),
pw AS (SELECT DISTINCT w FROM seated),
-- work-level edges: the part's work -> its parent's work, two different works (nearly every part already sits on its
-- dataset's work since the 2026-10-23 wave, so this is small). A work holding a dataset and its files moves only on the
-- dataset's own declaration (a sub-dataset IsPartOf an outer dataset), never because one of its files names a second
-- parent (a Chandra observation in two data collections, a WDCC variable naming another experiment).
e0 AS (
  SELECT DISTINCT rp.c, rp.p
  FROM rp
  LEFT JOIN pw ON pw.w = rp.c
  LEFT JOIN seated sp ON sp.w = rp.c AND sp.parent_doi = rp.rec_doi
  WHERE rp.c <> rp.p AND (pw.w IS NULL OR sp.w IS NOT NULL)
),
ew AS (SELECT c AS id FROM e0 UNION SELECT p FROM e0),
-- both works live: in openalex_works and not merged away (checked for the edges' works only)
live AS (
  SELECT w.id FROM openalex.works.openalex_works w
  LEFT SEMI JOIN ew ON ew.id = w.id
  LEFT ANTI JOIN openalex.works.merged_work_ids m ON m.loser_work_id = w.id
),
e AS (SELECT e0.c, e0.p FROM e0 JOIN live lc ON lc.id = e0.c JOIN live lp ON lp.id = e0.p),
outdeg AS (SELECT c, COUNT(*) AS n FROM e GROUP BY c),
up AS (SELECT e.c, e.p FROM e JOIN outdeg o ON o.c = e.c WHERE o.n = 1),  -- single-parent steps
s AS (
  SELECT u1.c, u1.p AS p1, u2.p AS p2, u3.p AS p3, u4.p AS p4, u5.p AS p5, u6.p AS p6
  FROM up u1 LEFT JOIN up u2 ON u2.c = u1.p LEFT JOIN up u3 ON u3.c = u2.p LEFT JOIN up u4 ON u4.c = u3.p
  LEFT JOIN up u5 ON u5.c = u4.p LEFT JOIN up u6 ON u6.c = u5.p
),
-- one parent: the top of the chain; a chain of 6+ steps or a loop is held
single AS (
  SELECT c AS loser_work_id, COALESCE(p5, p4, p3, p2, p1) AS winner_work_id,
         concat('part_of_depth', CASE WHEN p2 IS NULL THEN 1 WHEN p3 IS NULL THEN 2 WHEN p4 IS NULL THEN 3 WHEN p5 IS NULL THEN 4 ELSE 5 END) AS signals,
         CASE WHEN p6 IS NOT NULL OR p2 <=> c OR p3 <=> c OR p4 <=> c OR p5 <=> c THEN 'deep_or_loop' END AS hold
  FROM s
),
top1 AS (SELECT loser_work_id AS n, winner_work_id AS t FROM single WHERE hold IS NULL),
-- two or more parents: each followed to its top; one top merges there, several are held
mtops AS (
  SELECT DISTINCT e.c AS m, COALESCE(t1.t, e.p) AS t
  FROM e JOIN outdeg o ON o.c = e.c AND o.n > 1 LEFT JOIN top1 t1 ON t1.n = e.p
),
mres AS (SELECT m, MIN(t) AS t, COUNT(DISTINCT t) AS n_tops FROM mtops GROUP BY m),
fin AS (
  SELECT s1.loser_work_id, CASE WHEN s1.hold IS NULL THEN COALESCE(r.t, s1.winner_work_id) ELSE s1.winner_work_id END AS winner_work_id,
         s1.signals, s1.hold
  FROM single s1 LEFT JOIN mres r ON r.m = s1.winner_work_id AND r.n_tops = 1
  UNION ALL
  SELECT r.m, COALESCE(r2.t, r.t), CASE WHEN r.n_tops = 1 THEN 'part_of_multi_one_top' ELSE 'part_of_multi' END,
         CASE WHEN r.n_tops > 1 THEN 'multi_parent' END
  FROM mres r LEFT JOIN mres r2 ON r2.m = r.t AND r2.n_tops = 1
)
SELECT loser_work_id, winner_work_id, signals, hold FROM fin WHERE winner_work_id <> loser_work_id

In [ ]:
CREATE OR REPLACE TABLE openalex.works.part_of_merge_losers AS
WITH p AS (SELECT * FROM openalex.works.part_of_merge_pairs),
ex AS (SELECT * FROM p WHERE hold IS NULL),
winners AS (SELECT DISTINCT winner_work_id FROM ex),
members AS (SELECT winner_work_id, loser_work_id AS work_id FROM ex UNION SELECT winner_work_id, winner_work_id FROM ex),
-- the losers' record keys, as MapWorkIds will look them up after the unpin
lrec AS (
  SELECT x.winner_work_id, x.loser_work_id, l.merge_key AS mk
  FROM ex x
  JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
  JOIN openalex.works.locations_w_types l
    ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
),
lkeys AS (
  SELECT DISTINCT x.winner_work_id, x.loser_work_id, kv.col, kv.k
  FROM lrec x LATERAL VIEW explode(map('doi', NULLIF(x.mk.doi, ''), 'pmid', NULLIF(x.mk.pmid, ''),
                                       'arxiv', CASE WHEN x.mk.arxiv IN ('arXiv:', '') THEN NULL ELSE x.mk.arxiv END,
                                       'title_author', NULLIF(x.mk.title_author, ''))) kv AS col, k
  WHERE kv.k IS NOT NULL
),
keyless AS (
  SELECT DISTINCT loser_work_id FROM lrec
  WHERE NULLIF(mk.doi, '') IS NULL AND NULLIF(mk.pmid, '') IS NULL AND (mk.arxiv IS NULL OR mk.arxiv IN ('arXiv:', ''))
),
kin AS (SELECT DISTINCT col, k FROM lkeys),
holders AS (
  SELECT 'doi' AS col, m.doi AS k, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'doi' AND kin.k = m.doi
  UNION ALL SELECT 'pmid', m.pmid, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'pmid' AND kin.k = m.pmid
  UNION ALL SELECT 'arxiv', m.arxiv, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'arxiv' AND kin.k = m.arxiv
  UNION ALL SELECT 'title_author', m.title_author, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'title_author' AND kin.k = m.title_author
),
-- a key held by a work outside the winner and its losers; a shared title key only counts for a keyless loser record
shared AS (
  SELECT DISTINCT lk.loser_work_id FROM lkeys lk JOIN holders h ON h.col = lk.col AND h.k = lk.k
  WHERE NOT EXISTS (SELECT 1 FROM members m WHERE m.winner_work_id = lk.winner_work_id AND m.work_id = h.id)
    AND (lk.col <> 'title_author' OR EXISTS (SELECT 1 FROM keyless kl WHERE kl.loser_work_id = lk.loser_work_id))
)
SELECT p.winner_work_id, p.loser_work_id, p.signals,
       COALESCE(p.hold,
                CASE WHEN w.winner_work_id IS NOT NULL THEN 'chained'
                     WHEN sh.loser_work_id IS NOT NULL THEN 'loser_key_shared' END) AS hold_reason,
       current_timestamp() AS run_ts
FROM p LEFT JOIN winners w ON w.winner_work_id = p.loser_work_id LEFT JOIN shared sh ON sh.loser_work_id = p.loser_work_id

In [ ]:
SELECT signals, hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.part_of_merge_losers GROUP BY 1, 2 ORDER BY 3 DESC

## All classes: one row per loser, the cross-class hold

In [ ]:
CREATE OR REPLACE TABLE openalex.works.declared_link_losers AS
WITH u AS (
  SELECT 'preprint' AS link, group_work_id, winner_work_id, loser_work_id, n_pairs, signals, hold_reason, run_ts
  FROM openalex.works.preprint_link_merge_losers
  UNION ALL
  SELECT 'part_of', winner_work_id, winner_work_id, loser_work_id, 1, signals, hold_reason, run_ts
  FROM openalex.works.part_of_merge_losers
),
ex AS (SELECT link, winner_work_id, loser_work_id FROM u WHERE hold_reason IS NULL),
-- a work that is a loser or a winner in two classes' executable rows the same night
cc AS (
  SELECT id FROM (SELECT link, winner_work_id AS id FROM ex UNION SELECT link, loser_work_id FROM ex)
  GROUP BY id HAVING COUNT(DISTINCT link) > 1
),
-- held with its whole group (a preprint group, or a dataset's losers)
cg AS (
  SELECT DISTINCT u.link, u.group_work_id FROM u JOIN cc ON cc.id = u.loser_work_id OR cc.id = u.winner_work_id
)
SELECT u.link, u.group_work_id, u.winner_work_id, u.loser_work_id, u.n_pairs, u.signals,
       COALESCE(u.hold_reason, CASE WHEN cg.group_work_id IS NOT NULL THEN 'cross_class' END) AS hold_reason, u.run_ts
FROM u LEFT JOIN cg ON cg.link = u.link AND cg.group_work_id = u.group_work_id

In [ ]:
SELECT link, hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.declared_link_losers GROUP BY 1, 2 ORDER BY 1, 3 DESC

## Tripwire and invariants (before any write)

In [ ]:
WITH x AS (SELECT * FROM openalex.works.declared_link_losers WHERE hold_reason IS NULL),
s AS (SELECT COUNT(*) AS n, COUNT(DISTINCT loser_work_id) AS n_losers,
             COUNT_IF(link = 'preprint') AS n_preprint, COUNT_IF(link = 'part_of') AS n_part_of FROM x),
c AS (SELECT COUNT(*) AS loser_is_winner FROM x JOIN (SELECT DISTINCT winner_work_id FROM x) w ON w.winner_work_id = x.loser_work_id)
SELECT CASE
  WHEN s.n_preprint > 100000 THEN RAISE_ERROR(concat('MergeDeclaredLinks: ', s.n_preprint, ' executable preprint losers > 100,000 tripwire; nothing written'))
  WHEN s.n_part_of > 50000 THEN RAISE_ERROR(concat('MergeDeclaredLinks: ', s.n_part_of, ' executable part_of losers > 50,000 tripwire; nothing written'))
  WHEN s.n_losers <> s.n THEN RAISE_ERROR('MergeDeclaredLinks: a loser has two winners')
  WHEN c.loser_is_winner > 0 THEN RAISE_ERROR('MergeDeclaredLinks: a loser is also a winner')
END AS tripwire
FROM s CROSS JOIN c

## Execute, once for all classes
Audit first, aliases before the unpin (the aliases are read from the losers' pinned records), map rows before pins.

In [ ]:
INSERT INTO openalex.works.declared_link_merges_audit
SELECT x.run_ts, x.link, 'pin', x.loser_work_id, x.winner_work_id,
       r.provenance, r.native_id_namespace, r.native_id, r.work_id_source, r.openalex_created_dt, r.openalex_updated_dt, r.seeded_dt, r.seeded_from,
       NULL, NULL, NULL, NULL, NULL, NULL
FROM openalex.works.declared_link_losers x
JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
WHERE x.hold_reason IS NULL

In [ ]:
INSERT INTO openalex.works.declared_link_merges_audit
SELECT x.run_ts, x.link, 'map', x.loser_work_id, x.winner_work_id,
       NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       m.doi, m.pmid, m.arxiv, m.title_author, m.created_date, m.updated_date
FROM openalex.works.declared_link_losers x
JOIN openalex.works.work_id_map m ON m.id = x.loser_work_id
WHERE x.hold_reason IS NULL

In [ ]:
CREATE OR REPLACE TABLE openalex.works.declared_link_merge_aliases AS
SELECT DISTINCT x.run_ts, x.link, x.loser_work_id, x.winner_work_id AS id, NULLIF(l.merge_key.doi, '') AS doi, l.merge_key.pmid AS pmid,
       l.merge_key.arxiv AS arxiv, l.merge_key.title_author AS title_author
FROM openalex.works.declared_link_losers x
JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
JOIN openalex.works.locations_w_types l
  ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
WHERE x.hold_reason IS NULL
  AND COALESCE(NULLIF(l.merge_key.doi, ''), l.merge_key.pmid, l.merge_key.arxiv, l.merge_key.title_author) IS NOT NULL

In [ ]:
INSERT INTO openalex.works.work_id_map (id, doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT DISTINCT id, doi, pmid, arxiv, title_author, current_date(), current_timestamp()
FROM openalex.works.declared_link_merge_aliases x
WHERE NOT EXISTS (SELECT 1 FROM openalex.works.work_id_map m WHERE m.id = x.id AND m.doi <=> x.doi AND m.pmid <=> x.pmid
                  AND m.arxiv <=> x.arxiv AND m.title_author <=> x.title_author)

In [ ]:
INSERT INTO openalex.works.declared_link_merges_audit
SELECT run_ts, link, 'alias', loser_work_id, id, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       doi, pmid, arxiv, title_author, current_date(), NULL
FROM openalex.works.declared_link_merge_aliases

In [ ]:
DELETE FROM openalex.works.work_id_map m
WHERE EXISTS (SELECT 1 FROM openalex.works.declared_link_losers x WHERE x.hold_reason IS NULL AND x.loser_work_id = m.id)

In [ ]:
DELETE FROM openalex.works.location_work_ids r
WHERE EXISTS (SELECT 1 FROM openalex.works.declared_link_losers x WHERE x.hold_reason IS NULL AND x.loser_work_id = r.work_id)

In [ ]:
MERGE INTO openalex.works.merged_work_ids m
USING (SELECT DISTINCT loser_work_id, winner_work_id, run_ts,
              CASE link WHEN 'preprint' THEN 'preprint_link_nightly' WHEN 'part_of' THEN 'part_of_nightly' END AS source
       FROM openalex.works.declared_link_losers WHERE hold_reason IS NULL) s
ON m.loser_work_id = s.loser_work_id AND m.winner_work_id = s.winner_work_id
WHEN NOT MATCHED THEN INSERT (loser_work_id, winner_work_id, merged_at, source)
VALUES (s.loser_work_id, s.winner_work_id, s.run_ts, s.source)

In [ ]:
INSERT INTO openalex.works.declared_link_merges
SELECT run_ts, link, loser_work_id, winner_work_id, group_work_id, n_pairs, signals
FROM openalex.works.declared_link_losers WHERE hold_reason IS NULL

## Citations follow the merge
Rows citing a ledgered loser move to its winner, and the loser's own reference rows come along (they keep their
`native_id` + `ref_ind`, so they never collide with the winner's). Idempotent: only rows still keyed to a loser move.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW declared_link_repoint AS
SELECT p.loser_work_id, p.winner_work_id
FROM (SELECT loser_work_id, MIN(winner_work_id) AS winner_work_id
      FROM (SELECT loser_work_id, winner_work_id FROM openalex.works.preprint_link_merges
            UNION ALL
            SELECT loser_work_id, winner_work_id FROM openalex.works.declared_link_merges)
      GROUP BY loser_work_id) p
WHERE NOT EXISTS (SELECT 1 FROM openalex.works.location_work_ids r WHERE r.work_id = p.loser_work_id)

In [ ]:
INSERT INTO openalex.works.declared_link_merges_refs_audit
SELECT current_timestamp(), 'cited', r.citing_work_id, r.native_id, r.native_id_namespace, r.ref_ind, r.cited_work_id, p.winner_work_id
FROM openalex.works.work_references r JOIN declared_link_repoint p ON r.cited_work_id = p.loser_work_id
UNION ALL
SELECT current_timestamp(), 'citing', r.citing_work_id, r.native_id, r.native_id_namespace, r.ref_ind, r.citing_work_id, p.winner_work_id
FROM openalex.works.work_references r JOIN declared_link_repoint p ON r.citing_work_id = p.loser_work_id

In [ ]:
MERGE INTO openalex.works.work_references r
USING declared_link_repoint p ON r.cited_work_id = p.loser_work_id
WHEN MATCHED THEN UPDATE SET r.cited_work_id = p.winner_work_id, r.updated_timestamp = current_timestamp()

In [ ]:
MERGE INTO openalex.works.work_references r
USING declared_link_repoint p ON r.citing_work_id = p.loser_work_id
WHEN MATCHED THEN UPDATE SET r.citing_work_id = p.winner_work_id, r.updated_timestamp = current_timestamp()

In [ ]:
WITH x AS (
  SELECT link, COUNT(*) AS losers_tonight, COUNT_IF(hold_reason IS NULL) AS losers_merged,
         COUNT(DISTINCT CASE WHEN hold_reason IS NULL THEN winner_work_id END) AS winners
  FROM openalex.works.declared_link_losers GROUP BY link
),
a AS (SELECT link, COUNT(*) AS alias_rows FROM openalex.works.declared_link_merge_aliases GROUP BY link),
h AS (SELECT link, COUNT(*) AS losers_merged_all_time FROM openalex.works.declared_link_merges GROUP BY link)
SELECT x.link, x.losers_tonight, x.losers_merged, x.winners, COALESCE(a.alias_rows, 0) AS alias_rows,
       COALESCE(h.losers_merged_all_time, 0) AS losers_merged_all_time
FROM x LEFT JOIN a ON a.link = x.link LEFT JOIN h ON h.link = x.link ORDER BY x.link